In [1]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ── Motor Parameters (Healthy) ──────────────────────────
params = {
    'R'  : 1.0,
    'L'  : 0.01,
    'Ke' : 0.01,
    'Kt' : 0.01,
    'J'  : 0.01,
    'B'  : 0.001,
    'V'  : 12.0,
    'TL' : 0.05,
}

# ── Physics Function ─────────────────────────────────────
def motor_derivatives(t, state, params):
    i     = state[0]
    omega = state[1]

    R  = params['R']
    L  = params['L']
    Ke = params['Ke']
    Kt = params['Kt']
    J  = params['J']
    B  = params['B']
    V  = params['V']
    TL = params['TL']

    if t >= 2.0:
        TL = TL * 1.05

    di_dt     = (V - R*i - Ke*omega) / L
    domega_dt = (Kt*i - B*omega - TL) / J

    return [di_dt, domega_dt]

# ── Run Simulation ───────────────────────────────────────
t_span = (0, 5)
t_eval = np.linspace(0, 5, 5000)
initial_state = [0.0, 0.0]

solution = solve_ivp(
    fun=motor_derivatives,
    t_span=t_span,
    y0=initial_state,
    t_eval=t_eval,
    args=(params,),
    method='RK45',
    rtol=1e-6,
    atol=1e-8
)

time    = solution.t
current = solution.y[0]
speed   = solution.y[1]

# ── Plot and Save ────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6))

ax1.plot(time, current, color='royalblue', linewidth=1.5)
ax1.axvline(x=2.0, color='red', linestyle='--', label='Stimulus at t=2s')
ax1.set_ylabel('Current (A)')
ax1.set_title('DC Motor Response to 5% Load Step')
ax1.legend()
ax1.grid(True)

ax2.plot(time, speed, color='darkorange', linewidth=1.5)
ax2.axvline(x=2.0, color='red', linestyle='--', label='Stimulus at t=2s')
ax2.set_ylabel('Speed (rad/s)')
ax2.set_xlabel('Time (s)')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('../plots/healthy_motor_response.png', dpi=150)
plt.close()
print("Simulation complete. Plot saved to plots folder.")

Simulation complete. Plot saved to plots folder.


In [ ]:
# Stage 2 — compare the healthy motor with representative degraded motors
def run_motor_simulation(sim_params):
    """Run one 5-second motor simulation with the standard load-step stimulus."""
    result = solve_ivp(
        fun=motor_derivatives,
        t_span=(0, 5),
        y0=[0.0, 0.0],
        t_eval=np.linspace(0, 5, 5000),
        args=(sim_params,),
        method='RK45',
        rtol=1e-6,
        atol=1e-8
    )
    if not result.success:
        raise RuntimeError(f'Motor simulation failed: {result.message}')
    return result.t, result.y[0], result.y[1]

fault_parameters = {
    'Healthy': params.copy(),
    'Bearing fault (B x2.0)': {**params, 'B': params['B'] * 2.0},
    'Winding fault (R x1.5)': {**params, 'R': params['R'] * 1.5},
    'Mechanical fault (J x1.3)': {**params, 'J': params['J'] * 1.3},
}

responses = {
    label: run_motor_simulation(sim_params)
    for label, sim_params in fault_parameters.items()
}

fig, (ax_current, ax_speed) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
colours = ['black', 'crimson', 'royalblue', 'darkorange']
for (label, (sim_time, sim_current, sim_speed)), colour in zip(responses.items(), colours):
    ax_current.plot(sim_time, sim_current, label=label, color=colour, linewidth=1.4)
    ax_speed.plot(sim_time, sim_speed, label=label, color=colour, linewidth=1.4)

for axis in (ax_current, ax_speed):
    axis.axvline(2.0, color='gray', linestyle='--', linewidth=1, label='5% load step')
    axis.grid(True, alpha=0.3)
    axis.legend(loc='best')

ax_current.set_title('Healthy and Degraded DC Motor Responses')
ax_current.set_ylabel('Current (A)')
ax_speed.set_ylabel('Speed (rad/s)')
ax_speed.set_xlabel('Time (s)')
plt.tight_layout()
plt.savefig('../plots/healthy_vs_degraded_responses.png', dpi=150, bbox_inches='tight')
plt.close()
print('Stage 2 complete. Saved plots/healthy_vs_degraded_responses.png')


In [ ]:
# Stage 3 — extract a transient-response fingerprint from t >= 2.0 s
from scipy.stats import kurtosis

FEATURE_NAMES = [
    'rpm_drop_pct', 'settling_time_s', 'current_peak_a', 'current_rms_a',
    'current_rise_pct', 'speed_rms_rad_s', 'overshoot_pct',
    'speed_kurtosis', 'current_kurtosis', 'current_crest_factor'
]

def extract_fingerprint(time, current, speed, stimulus_time=2.0):
    """Return ten response features using only the post-stimulus samples.

    The first post-stimulus sample is the reference state.  The final 10%
    of the window estimates steady state; settling uses a 2% band around it.
    """
    time = np.asarray(time)
    current = np.asarray(current)
    speed = np.asarray(speed)
    post_mask = time >= stimulus_time
    if post_mask.sum() < 10:
        raise ValueError('At least ten post-stimulus samples are required.')

    t = time[post_mask]
    i = current[post_mask]
    w = speed[post_mask]
    i_initial, w_initial = i[0], w[0]
    tail_start = max(1, int(0.9 * len(t)))
    i_final = np.mean(i[tail_start:])
    w_final = np.mean(w[tail_start:])

    # Positive RPM drop means that speed dipped below its pre-step value.
    rpm_drop_pct = 100 * max(0.0, (w_initial - np.min(w)) / max(abs(w_initial), 1e-12))
    current_peak_a = np.max(np.abs(i))
    current_rms_a = np.sqrt(np.mean(i ** 2))
    current_rise_pct = 100 * max(0.0, (np.max(i) - i_initial) / max(abs(i_initial), 1e-12))
    speed_rms_rad_s = np.sqrt(np.mean(w ** 2))

    # Measure overshoot in the direction of the net speed response.
    speed_change = w_final - w_initial
    if abs(speed_change) < 1e-12:
        overshoot_pct = 0.0
        settling_time_s = 0.0
    else:
        direction = np.sign(speed_change)
        overshoot_pct = 100 * max(0.0, np.max(direction * (w - w_final)) / abs(speed_change))
        tolerance = 0.02 * abs(speed_change)
        outside_band = np.abs(w - w_final) > tolerance
        settling_index = np.where(outside_band)[0][-1] + 1 if np.any(outside_band) else 0
        settling_time_s = (t[min(settling_index, len(t) - 1)] - stimulus_time)

    fingerprint = {
        'rpm_drop_pct': rpm_drop_pct,
        'settling_time_s': settling_time_s,
        'current_peak_a': current_peak_a,
        'current_rms_a': current_rms_a,
        'current_rise_pct': current_rise_pct,
        'speed_rms_rad_s': speed_rms_rad_s,
        'overshoot_pct': overshoot_pct,
        'speed_kurtosis': kurtosis(w, fisher=False, bias=False),
        'current_kurtosis': kurtosis(i, fisher=False, bias=False),
        'current_crest_factor': current_peak_a / max(current_rms_a, 1e-12),
    }
    return fingerprint

# Inspect fingerprints for the four Stage 2 response types.
fingerprints = {
    label: extract_fingerprint(sim_time, sim_current, sim_speed)
    for label, (sim_time, sim_current, sim_speed) in responses.items()
}
for label, fingerprint in fingerprints.items():
    print(f'\n{label}')
    for feature in FEATURE_NAMES:
        print(f'  {feature}: {fingerprint[feature]:.5f}')
